# 第13章 RAG

## 13.3 RAG 向けに LLM を指示チューニングする

### 13.3.2 指示チューニングしたモデルを LangChain で使う

#### 環境の準備

In [1]:
!pip install 'datasets<4.0.0' transformers[torch,sentencepiece] langchain langchain-community langchain-huggingface faiss-cpu jq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling d

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
from transformers.trainer_utils import set_seed
set_seed(42)

#### Chat Model の作成

In [4]:
import torch
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFacePipeline,
)
from transformers import(
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja-aio-retriever"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

generation_config = {
    "max_new_tokens": 32,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
}

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

chat_model = ChatHuggingFace(llm=llm, tokenizer=tokenizer)

config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/914k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.30M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


#### Embedding Model の作成

In [5]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = "BAAI/bge-m3"

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model,
    model_kwargs={"model_kwargs": {"torch_dtype": torch.bfloat16}},
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

#### データストアの構築

In [6]:
# 検索対象の文書集合のファイルをダウンロード
!wget  \
https://github.com/ghmagazine/llm-book/raw/main/chapter13/docs.json

--2026-06-14 10:41:38--  https://github.com/ghmagazine/llm-book/raw/main/chapter13/docs.json
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ghmagazine/llm-book/main/chapter13/docs.json [following]
--2026-06-14 10:41:38--  https://raw.githubusercontent.com/ghmagazine/llm-book/main/chapter13/docs.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1178382 (1.1M) [text/plain]
Saving to: ‘docs.json’

docs.json           100%[===================>]   1.12M  --.-KB/s    in 0.02s   

2026-06-14 10:41:38 (58.6 MB/s) - ‘docs.json’ saved [1178382/1178382]



In [7]:
from langchain_community.document_loaders import JSONLoader

document_loader = JSONLoader(
    file_path="./docs.json",
    jq_schema=".text",
    json_lines=True,
)

documents = document_loader.load()

In [8]:
print(len(documents))

103


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
    add_start_index=True,
)

split_documents = text_splitter.split_documents(documents)

In [10]:
print(len(split_documents))

1475


#### 検索対象の文書のベクトルインデックスの作成

In [11]:
from langchain_community.vectorstores import FAISS

vectorstores = FAISS.from_documents(split_documents, embedding_model)

In [12]:
print(vectorstores.index.ntotal)

1475


#### Retrieverコンポーネントの作成

In [13]:
retriever = vectorstores.as_retriever(search_kwargs={"k": 3})

In [14]:
from pprint import pprint

retrieved_documents = retriever.invoke("四国地方で一番高い山は？")

pprint(retrieved_documents)

[Document(id='364ebc3a-2739-49a9-816e-a4b34ec1f7f5', metadata={'source': '/content/docs.json', 'seq_num': 26, 'start_index': 0}, page_content='この項目に含まれる文字「鎚」は、オペレーティングシステムやブラウザなどの環境により表示が異なります。 石鎚山（いしづちさん、いしづちやま）は、四国山地西部に位置する標高1,982 mの山で、近畿以西を「西日本」とした場合の西日本最高峰で、山頂から望む展望が四国八十八景64番に選定。愛媛県西条市と久万高原町の境界に位置する。 石鉄山、石鈇山、石土山、石槌山とも表記され、伊予の高嶺とも呼ばれる。『日本霊異記』には「石槌山」と記され、延喜式の神名帳（延喜式神名帳）では「石鉄神社」と記されている。前神寺および横峰寺では「石鈇山（しゃくまざん）」とも呼ぶ。'),
 Document(id='cef4f4e8-65e4-4d88-9b43-8501ef77aa3c', metadata={'source': '/content/docs.json', 'seq_num': 1, 'start_index': 0}, page_content='富士山（ふじさん）は、静岡県（富士宮市、富士市、裾野市、御殿場市、駿東郡小山町）と山梨県（富士吉田市、南都留郡鳴沢村）に跨る活火山である。標高3776.12 m、日本最高峰（剣ヶ峰）の独立峰で、その優美な風貌は日本国外でも日本の象徴として広く知られている。 数多くの芸術作品の題材とされ芸術面のみならず、気候や地層など地質学的にも社会に大きな影響を与えている。懸垂曲線の山容を有した玄武岩質成層火山で構成され、その山体は駿河湾の海岸まで及ぶ。'),
 Document(id='b8efa318-bff9-43c2-b1de-098fb6a087b1', metadata={'source': '/content/docs.json', 'seq_num': 96, 'start_index': 0}, page_content='四阿山（あずまやさん）は、長野県と群馬県の県境に跨る山。標高2,354 m。日本百名山の一つに数えられている。吾妻山・吾嬬山（あがつま

#### RAGのChainの構築と実行

In [15]:
from langchain_core.prompts import ChatPromptTemplate

# 任意のqueryからメッセージを構築するPrompt Templateを作成
rag_prompt_text = (
    "あなたには今からクイズに答えてもらいます。"
    "問題を与えますので、その解答のみを簡潔に出力してください。\n"
    "また解答の参考になりうるテキストを与えます。"
    "解答を含まない場合もあるのでその場合は無視してください。\n\n"
    "---\n{context}\n---\n\n問題: {query}"
)

rag_prompt_template = ChatPromptTemplate.from_messages(
    [("user", rag_prompt_text)]
)

In [16]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

def format_documents_func(documents: list[Document]) -> str:
    """文書のリストを改行で連結した一つの文字として返す"""
    return "\n\n".join(
        document.page_content for document in documents
    )

format_documents = RunnableLambda(format_documents_func)

In [17]:
from langchain_core.prompt_values import ChatPromptValue

def chat_model_resp_only_func(
    chat_prompt_value: ChatPromptValue,
) -> str:
    """
    chat_modelにchat_prompt_valueを入力し、
    出力からモデルの応答部分のみを文字列で返す
    """
    chat_prompt = chat_model._to_chat_prompt(
        chat_prompt_value.messages
    )
    chat_output_message = chat_model.invoke(chat_prompt_value)
    response_text = chat_output_message.content[len(chat_prompt) :]
    return response_text

chat_model_resp_only = RunnableLambda(
    chat_model_resp_only_func
)

In [18]:
from langchain_core.runnables import RunnablePassthrough

# RAGの一連の処理を行うChainを作成
rag_chain = (
    {
        "context": retriever | format_documents,
        "query": RunnablePassthrough(),
    }
    | rag_prompt_template
    | chat_model_resp_only
)

In [19]:
# Chainを実行し、結果を確認
rag_chain_output = rag_chain.invoke("四国地方で一番高い山は？")
print(rag_chain_output)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


石鎚山
